In [ ]:
!pip -q install sentence-transformers openai pandas numpy scikit-learn

In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
documents = {
    "doc1": """
    Machine learning is a branch of artificial intelligence.
    It allows computers to learn patterns from data without
    being explicitly programmed. Machine learning can be used
    for classification, regression, and prediction tasks.
    """,

    "doc2": """
    Deep learning is a subset of machine learning that uses
    artificial neural networks with multiple layers.
    Deep learning is widely used in image recognition,
    speech recognition, and natural language processing.
    """,

    "doc3": """
    Natural language processing, or NLP, is a field of
    artificial intelligence that focuses on enabling computers
    to understand, process, and generate human language.
    NLP is used in translation, chatbots, and text classification.
    """,

    "doc4": """
    Computer vision is an area of artificial intelligence
    that enables computers to understand and analyze images
    and videos. Convolutional neural networks are commonly
    used for image classification and object detection.
    """,

    "doc5": """
    Retrieval Augmented Generation, known as RAG, combines
    information retrieval with large language models.
    The retrieval system searches a knowledge base for
    relevant information before the language model generates
    an answer.
    """
}

print("Number of documents:", len(documents))

Number of documents: 5


In [ ]:
chunks = []

for doc_id, text in documents.items():
    chunks.append({
        "chunk_id": f"{doc_id}_chunk_1",
        "document_id": doc_id,
        "text": text.strip()
    })

chunks_df = pd.DataFrame(chunks)

print(chunks_df)

       chunk_id document_id                                               text
0  doc1_chunk_1        doc1  Machine learning is a branch of artificial int...
1  doc2_chunk_1        doc2  Deep learning is a subset of machine learning ...
2  doc3_chunk_1        doc3  Natural language processing, or NLP, is a fiel...
3  doc4_chunk_1        doc4  Computer vision is an area of artificial intel...
4  doc5_chunk_1        doc5  Retrieval Augmented Generation, known as RAG, ...


In [ ]:
queries = [
    {
        "query_id": "q1",
        "query": "What is machine learning?",
        "relevant_chunks": ["doc1_chunk_1"]
    },

    {
        "query_id": "q2",
        "query": "What does deep learning use?",
        "relevant_chunks": ["doc2_chunk_1"]
    },

    {
        "query_id": "q3",
        "query": "What is natural language processing?",
        "relevant_chunks": ["doc3_chunk_1"]
    },

    {
        "query_id": "q4",
        "query": "What is computer vision used for?",
        "relevant_chunks": ["doc4_chunk_1"]
    },

    {
        "query_id": "q5",
        "query": "How does RAG work?",
        "relevant_chunks": ["doc5_chunk_1"]
    }
]

queries_df = pd.DataFrame(queries)

queries_df

,query_id,query,relevant_chunks
0,q1,What is machine learning?,[doc1_chunk_1]
1,q2,What does deep learning use?,[doc2_chunk_1]
2,q3,What is natural language processing?,[doc3_chunk_1]
3,q4,What is computer vision used for?,[doc4_chunk_1]
4,q5,How does RAG work?,[doc5_chunk_1]


In [ ]:
model = SentenceTransformer("BAAI/bge-m3")

print("BGE-M3 loaded successfully!")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

BGE-M3 loaded successfully!


In [ ]:
document_texts = chunks_df["text"].tolist()

start_time = time.time()

document_embeddings = model.encode(
    document_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

embedding_time = time.time() - start_time

print("Embedding shape:", document_embeddings.shape)
print("Embedding time:", embedding_time, "seconds")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Embedding shape: (5, 1024)
Embedding time: 0.9138245582580566 seconds


In [ ]:
query_texts = queries_df["query"].tolist()

start_time = time.time()

query_embeddings = model.encode(
    query_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

query_time = time.time() - start_time

print("Query embedding shape:", query_embeddings.shape)
print("Query embedding time:", query_time, "seconds")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Query embedding shape: (5, 1024)
Query embedding time: 0.0964195728302002 seconds


In [ ]:
similarity_matrix = cosine_similarity(
    query_embeddings,
    document_embeddings
)

results = []

for i, query_row in queries_df.iterrows():

    scores = similarity_matrix[i]

    ranked_indices = np.argsort(scores)[::-1]

    for rank, idx in enumerate(ranked_indices, start=1):

        results.append({
            "query_id": query_row["query_id"],
            "query": query_row["query"],
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "score": scores[idx]
        })

results_df = pd.DataFrame(results)

results_df

,query_id,query,rank,chunk_id,score
0,q1,What is machine learning?,1,doc1_chunk_1,0.770675
1,q1,What is machine learning?,2,doc2_chunk_1,0.669581
2,q1,What is machine learning?,3,doc4_chunk_1,0.611845
3,q1,What is machine learning?,4,doc3_chunk_1,0.593180
4,q1,What is machine learning?,5,doc5_chunk_1,0.523881
5,q2,What does deep learning use?,1,doc2_chunk_1,0.746062
6,q2,What does deep learning use?,2,doc1_chunk_1,0.586890
7,q2,What does deep learning use?,3,doc3_chunk_1,0.552023
8,q2,What does deep learning use?,4,doc4_chunk_1,0.529942
9,q2,What does deep learning use?,5,doc5_chunk_1,0.527300


In [ ]:
def recall_at_k(results_df, queries_df, k):

    hits = 0

    for _, query in queries_df.iterrows():

        query_results = results_df[
            results_df["query_id"] == query["query_id"]
        ].sort_values("rank").head(k)

        retrieved = set(query_results["chunk_id"])

        relevant = set(query["relevant_chunks"])

        if retrieved.intersection(relevant):
            hits += 1

    return hits / len(queries_df)


recall_5 = recall_at_k(results_df, queries_df, 5)
recall_10 = recall_at_k(results_df, queries_df, 10)

print("Recall@5:", recall_5)
print("Recall@10:", recall_10)

Recall@5: 1.0
Recall@10: 1.0


In [ ]:
def mrr(results_df, queries_df):

    reciprocal_ranks = []

    for _, query in queries_df.iterrows():

        query_results = results_df[
            results_df["query_id"] == query["query_id"]
        ].sort_values("rank")

        relevant = set(query["relevant_chunks"])

        rank_found = None

        for _, row in query_results.iterrows():

            if row["chunk_id"] in relevant:
                rank_found = row["rank"]
                break

        if rank_found is not None:
            reciprocal_ranks.append(1 / rank_found)
        else:
            reciprocal_ranks.append(0)

    return np.mean(reciprocal_ranks)


mrr_score = mrr(results_df, queries_df)

print("MRR:", mrr_score)

MRR: 1.0


In [ ]:
!pip -q install -U openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 38.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.4/94.4 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 8.3 MB/s eta 0:00:00


In [ ]:
from getpass import getpass
import os

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

Enter your OpenAI API key: ··········


In [ ]:
from openai import OpenAI

client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

print("OpenAI client ready!")

OpenAI client ready!


In [ ]:
from sentence_transformers import SentenceTransformer

e5_model = SentenceTransformer("intfloat/multilingual-e5-base")

print("multilingual-e5-base loaded successfully!")

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/179k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.11GB            

...

tokenizer.json: reconstructing file:   0%|          |  0.00B / 17.1MB            

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

multilingual-e5-base loaded successfully!


In [ ]:
document_texts = chunks_df["text"].tolist()

start_time = time.time()

e5_document_embeddings = e5_model.encode(
    document_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

e5_document_time = time.time() - start_time

print("E5 document embedding shape:", e5_document_embeddings.shape)
print("E5 document embedding time:", e5_document_time, "seconds")


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

E5 document embedding shape: (5, 768)
E5 document embedding time: 0.06682372093200684 seconds


In [ ]:
query_texts = queries_df["query"].tolist()

start_time = time.time()

e5_query_embeddings = e5_model.encode(
    query_texts,
    normalize_embeddings=True,
    show_progress_bar=True
)

e5_query_time = time.time() - start_time

print("E5 query embedding shape:", e5_query_embeddings.shape)
print("E5 query embedding time:", e5_query_time, "seconds")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

E5 query embedding shape: (5, 768)
E5 query embedding time: 0.04365730285644531 seconds


In [ ]:
e5_similarity_matrix = cosine_similarity(
    e5_query_embeddings,
    e5_document_embeddings
)

e5_results = []

for i, query_row in queries_df.iterrows():

    scores = e5_similarity_matrix[i]

    ranked_indices = np.argsort(scores)[::-1]

    for rank, idx in enumerate(ranked_indices, start=1):

        e5_results.append({
            "query_id": query_row["query_id"],
            "query": query_row["query"],
            "rank": rank,
            "chunk_id": chunks_df.iloc[idx]["chunk_id"],
            "score": scores[idx]
        })

e5_results_df = pd.DataFrame(e5_results)

e5_results_df

,query_id,query,rank,chunk_id,score
0,q1,What is machine learning?,1,doc1_chunk_1,0.912034
1,q1,What is machine learning?,2,doc2_chunk_1,0.880334
2,q1,What is machine learning?,3,doc3_chunk_1,0.834234
3,q1,What is machine learning?,4,doc4_chunk_1,0.831256
4,q1,What is machine learning?,5,doc5_chunk_1,0.792004
5,q2,What does deep learning use?,1,doc2_chunk_1,0.889570
6,q2,What does deep learning use?,2,doc1_chunk_1,0.843596
7,q2,What does deep learning use?,3,doc4_chunk_1,0.812338
8,q2,What does deep learning use?,4,doc3_chunk_1,0.809033
9,q2,What does deep learning use?,5,doc5_chunk_1,0.777413


In [ ]:
e5_recall_5 = recall_at_k(
    e5_results_df,
    queries_df,
    5
)

e5_recall_10 = recall_at_k(
    e5_results_df,
    queries_df,
    10
)

print("E5 Recall@5:", e5_recall_5)
print("E5 Recall@10:", e5_recall_10)

E5 Recall@5: 1.0
E5 Recall@10: 1.0


In [ ]:
e5_mrr = mrr(
    e5_results_df,
    queries_df
)

print("E5 MRR:", e5_mrr)

E5 MRR: 1.0


In [ ]:
comparison = pd.DataFrame([
    {
        "Model": "BGE-M3",
        "Recall@5": recall_5,
        "Recall@10": recall_10,
        "MRR": mrr_score,
        "Query_Time_sec": query_time
    },
    {
        "Model": "multilingual-e5-base",
        "Recall@5": e5_recall_5,
        "Recall@10": e5_recall_10,
        "MRR": e5_mrr,
        "Query_Time_sec": e5_query_time
    }
])

comparison

,Model,Recall@5,Recall@10,MRR,Query_Time_sec
0,BGE-M3,1.0,1.0,1.0,0.096420
1,multilingual-e5-base,1.0,1.0,1.0,0.043657
